<a href="https://colab.research.google.com/github/EvnGgnn/TheGreenPixels/blob/Evan/Chlorophyll_Concentration_and_Shipping_Routes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data Collection Period: January 2021 - Present

At this link we can obtain chlorophyll density data from NASA Satellites
https://neo.gsfc.nasa.gov/view.php?datasetId=MY1DMW_CHLORA&date=2025-11-01

In [48]:
import numpy as np
import pandas as pd
import datetime
import os
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [49]:
FILEPATH = 'https://neo.gsfc.nasa.gov/archive/geotiff.float/MY1DMW_CHLORA/'
extension_list = []
#Generate the url for the geotiffs
initial_date = datetime.datetime(2015, 1, 1, 0, 0, 0)
end_week_date = initial_date+datetime.timedelta(days=+7)
while end_week_date <= datetime.datetime(2021, 2, 28, 0, 0, 0):
  extension_list.append(f"AQUA_MODIS.{initial_date.year:04d}{initial_date.month:02d}{initial_date.day:02d}_{end_week_date.year:04d}{end_week_date.month:02d}{end_week_date.day:02d}.L3m.8D.CHL.chlor_a.4km.tif")
  initial_date = end_week_date+datetime.timedelta(days=+1)
  end_week_date = initial_date+datetime.timedelta(days=+7)
  if end_week_date.year > initial_date.year:
    end_week_date = datetime.datetime(initial_date.year, 12, 31, 0, 0, 0)
extension_list

['AQUA_MODIS.20150101_20150108.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150109_20150116.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150117_20150124.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150125_20150201.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150202_20150209.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150210_20150217.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150218_20150225.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150226_20150305.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150306_20150313.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150314_20150321.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150322_20150329.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150330_20150406.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150407_20150414.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150415_20150422.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150423_20150430.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150501_20150508.L3m.8D.CHL.chlor_a.4km.tif',
 'AQUA_MODIS.20150509_20150516.L3m.8D.CH

Running the following codeblock will download all of the NASA files again,
make sure not to do this unless you want to

In [56]:
import requests
save_path = "/content/drive/MyDrive/Chlorophyll Project/chlorophyll_geotiff2"
os.makedirs(save_path, exist_ok=True)
for extension in extension_list:
  print(extension)
  r = requests.get(FILEPATH+extension)
  collab_path = os.path.join(save_path, extension)

  with open(collab_path, "wb") as f:
        f.write(r.content)

AQUA_MODIS.20150101_20150108.L3m.8D.CHL.chlor_a.4km.tif


KeyboardInterrupt: 

Once the Chlorophyll geotiff has been downloaded, this block can be run

In [ ]:
import rasterio
from rasterio.plot import show
#Run the following line if rioxarray can not be found
#%pip install rioxarray
import rioxarray
print("Finished Imports")

practice_path = '/content/drive/MyDrive/Chlorophyll Project/chlorophyll_geotiff/AQUA_MODIS.20150101_20150108.L3m.8D.CHL.chlor_a.4km.tif'
da = rioxarray.open_rasterio(practice_path, masked=True)

da = da.rio.reproject("EPSG:4326")
df = da[0].to_pandas()
df['Latitude'] = df.index # Save index as y coordinate
df = pd.melt(df, id_vars='Latitude', var_name='Longitude', value_name='mg/m3')
df = df.fillna(0)
display(df)

Now we need to get the data for cargo ships.

This works, but the file itself is around 9gbs. This takes up most drive storage, so download it to your local machine instead.

https://datacatalogfiles.worldbank.org/ddh-published/0037580/5/DR0045405/shipdensity_commercial_.zip

In [ ]:
import zipfile
MARITIME_PATH = 'https://datacatalogfiles.worldbank.org/ddh-published/0037580/5/DR0045405/shipdensity_commercial_.zip'
save_path = "/content/drive/MyDrive//Chlorophyll Project/maritime_geotiff"
os.makedirs(save_path, exist_ok=True)
r = requests.get(MARITIME_PATH)
collab_path = os.path.join(save_path, "Maritime_Density.zip")

with open(collab_path, "wb") as f:
      f.write(r.content)

%cd /content/drive/MyDrive//Chlorophyll Project/maritime_geotiff
!unzip Maritime_Density.zip


/content/drive/MyDrive/Chlorophyll Project/maritime_geotiff
Archive:  Maritime_Density.zip
replace ShipDensity_Commercial1.tif.ovr? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

Here we will create two visualizations for our data.

In [54]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive/Chlorophyll Project/chlorophyll_geotiff/'):
  for file in files:
    print("hello")

hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
